# Valencia Properties — Exploratory Data Analysis

Goal: understand the property market in Valencia Province using 8,761 cleaned listings from fotocasa.

**Sections**

1. Overview — what's in the data
2. Property types — what's for sale
3. Price distributions — how the market splits
4. Geography — where the action is
5. Cities & regions — best value vs prestige areas
6. Feature impact — does a pool actually move the price?
7. Distance effects — coast premium, capital premium
8. Listing freshness — how stale is the market?
9. Correlations — what predicts price
10. Key insights — summary findings

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

df = pd.read_parquet("../valencia_clean.parquet")
print(f"Loaded {len(df):,} listings, {df.shape[1]} columns")
df.head(3)

## 1. Overview

Quick stats and missing-value check on the cleaned dataset.

In [ ]:
summary = pd.DataFrame({
    "Metric": [
        "Total listings",
        "Unique cities",
        "Property subtypes",
        "Median price",
        "Median surface (m²)",
        "Median €/m²",
        "Coastal listings",
        "Date range (oldest listing, days)",
    ],
    "Value": [
        f"{len(df):,}",
        df["city"].nunique(),
        df["property_subtype"].nunique(),
        f"€{df['price_eur'].median():,.0f}",
        f"{df['surface_m2'].median():.0f}",
        f"€{df['price_per_m2'].median():,.0f}",
        f"{df['is_coastal'].sum():,} ({df['is_coastal'].mean()*100:.1f}%)",
        f"{df['listing_age_days'].max():.0f}",
    ],
})
summary

## 2. Property types

What kind of properties make up the Valencia market?

In [ ]:
subtype_counts = df["property_subtype"].value_counts().reset_index()
subtype_counts.columns = ["subtype", "count"]

fig = px.bar(
    subtype_counts,
    x="count",
    y="subtype",
    orientation="h",
    color="count",
    color_continuous_scale="Blues",
    title="Listings by property subtype",
    text="count",
)
fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=500, showlegend=False)
fig.update_traces(textposition="outside")
fig.show()

## 3. Price distributions

The price distribution is heavily right-skewed (many cheap properties, fewer luxury).
We use a log scale to make patterns visible.

In [ ]:
fig = make_subplots(rows=1, cols=3, subplot_titles=("Price (€)", "Surface (m²)", "€ / m²"))

fig.add_trace(go.Histogram(x=df["price_eur"], nbinsx=60, marker_color="#3b82f6"), row=1, col=1)
fig.add_trace(go.Histogram(x=df["surface_m2"], nbinsx=60, marker_color="#10b981"), row=1, col=2)
fig.add_trace(go.Histogram(x=df["price_per_m2"], nbinsx=60, marker_color="#f59e0b"), row=1, col=3)

fig.update_xaxes(type="log", row=1, col=1)
fig.update_xaxes(type="log", row=1, col=2)
fig.update_layout(height=400, title_text="Distributions (log x-axis on price & surface)", showlegend=False)
fig.show()

In [ ]:
fig = px.box(
    df,
    x="property_subtype",
    y="price_eur",
    color="property_type",
    title="Price by property subtype",
    log_y=True,
    color_discrete_map={"Casa": "#10b981", "Piso": "#3b82f6"},
)
fig.update_layout(height=500, xaxis_tickangle=-30)
fig.show()

## 4. Geographic distribution

Map of every listing, colored by €/m². Hovering shows price, surface, type.

In [ ]:
map_df = df.copy()
map_df["price_per_m2_capped"] = map_df["price_per_m2"].clip(upper=5000)

fig = px.scatter_mapbox(
    map_df,
    lat="latitude",
    lon="longitude",
    color="price_per_m2_capped",
    size="surface_m2",
    size_max=12,
    hover_data={
        "city": True,
        "price_eur": ":,.0f",
        "surface_m2": True,
        "bedrooms": True,
        "property_subtype": True,
        "price_per_m2_capped": False,
    },
    color_continuous_scale="Turbo",
    zoom=8,
    center={"lat": 39.4699, "lon": -0.3763},
    title="Property prices across Valencia Province (€/m², capped at 5000)",
    height=650,
)
fig.update_layout(mapbox_style="open-street-map", margin={"r": 0, "t": 40, "l": 0, "b": 0})
fig.show()

## 5. Cities & regions

Where is the inventory and what's it cost?

In [ ]:
city_stats = (
    df.groupby("city")
    .agg(
        listings=("id", "count"),
        median_price=("price_eur", "median"),
        median_ppm2=("price_per_m2", "median"),
        median_m2=("surface_m2", "median"),
    )
    .sort_values("listings", ascending=False)
    .head(20)
    .round(0)
    .astype({"median_price": int, "median_ppm2": int, "median_m2": int})
)
city_stats

In [ ]:
top_cities = city_stats.index.tolist()
fig = px.bar(
    city_stats.reset_index(),
    x="city",
    y="median_ppm2",
    color="listings",
    color_continuous_scale="Viridis",
    title="Median €/m² by city (top 20 by listing count)",
    text="median_ppm2",
    labels={"median_ppm2": "€/m²", "listings": "Listings"},
)
fig.update_traces(texttemplate="€%{text:,.0f}", textposition="outside")
fig.update_layout(height=500, xaxis_tickangle=-45)
fig.show()

In [ ]:
region_stats = df.groupby(["region", "property_type"]).agg(
    listings=("id", "count"),
    median_price=("price_eur", "median"),
    median_ppm2=("price_per_m2", "median"),
).reset_index()

fig = px.bar(
    region_stats,
    x="region",
    y="median_ppm2",
    color="property_type",
    barmode="group",
    text="median_ppm2",
    title="Median €/m² by region & property type",
    color_discrete_map={"Casa": "#10b981", "Piso": "#3b82f6"},
)
fig.update_traces(texttemplate="€%{text:,.0f}", textposition="outside")
fig.update_layout(height=450)
fig.show()

## 6. Feature impact on price

Each amenity boolean → does its presence move the price?

We compute the median €/m² uplift each feature provides, controlling for nothing
(this is descriptive, not causal — features correlate with each other).

In [ ]:
feature_cols = [
    "has_elevator", "has_parking", "has_terrace", "has_balcony",
    "has_pool", "has_garden", "has_air_conditioning", "has_heating",
    "has_storage", "is_furnished",
]

rows = []
for f in feature_cols:
    with_f = df[df[f]]["price_per_m2"].median()
    without_f = df[~df[f]]["price_per_m2"].median()
    uplift = with_f - without_f
    pct = (uplift / without_f) * 100 if without_f else 0
    rows.append({
        "feature": f.replace("has_", "").replace("is_", "").replace("_", " "),
        "median_with": with_f,
        "median_without": without_f,
        "uplift_eur_m2": uplift,
        "uplift_pct": pct,
        "n_with": df[f].sum(),
    })

impact = pd.DataFrame(rows).sort_values("uplift_pct", ascending=False)
impact

In [ ]:
fig = px.bar(
    impact,
    x="uplift_pct",
    y="feature",
    orientation="h",
    color="uplift_pct",
    color_continuous_scale="RdYlGn",
    color_continuous_midpoint=0,
    title="€/m² uplift (%) when feature is present",
    text="uplift_pct",
)
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(height=450, yaxis={"categoryorder": "total ascending"})
fig.show()

## 7. Distance effects

Does proximity to Valencia city or to the coast command a premium?

In [ ]:
df["dist_valencia_bin"] = pd.cut(
    df["distance_to_valencia_km"],
    bins=[0, 5, 10, 20, 30, 50, 200],
    labels=["0-5km", "5-10km", "10-20km", "20-30km", "30-50km", "50+km"],
)
df["dist_coast_bin"] = pd.cut(
    df["distance_to_coast_km"],
    bins=[0, 2, 5, 10, 20, 200],
    labels=["0-2km", "2-5km", "5-10km", "10-20km", "20+km"],
)

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    "€/m² vs distance to Valencia center",
    "€/m² vs distance to coast",
))

dist_v = df.groupby("dist_valencia_bin", observed=True)["price_per_m2"].median().reset_index()
dist_c = df.groupby("dist_coast_bin", observed=True)["price_per_m2"].median().reset_index()

fig.add_trace(go.Bar(
    x=dist_v["dist_valencia_bin"].astype(str),
    y=dist_v["price_per_m2"],
    marker_color="#3b82f6",
    text=dist_v["price_per_m2"].apply(lambda v: f"€{v:,.0f}"),
    textposition="outside",
), row=1, col=1)

fig.add_trace(go.Bar(
    x=dist_c["dist_coast_bin"].astype(str),
    y=dist_c["price_per_m2"],
    marker_color="#06b6d4",
    text=dist_c["price_per_m2"].apply(lambda v: f"€{v:,.0f}"),
    textposition="outside",
), row=1, col=2)

fig.update_layout(height=450, showlegend=False)
fig.show()

## 8. Listing freshness

How long has each listing been on the market? Lots of stale listings = soft market.

In [ ]:
age_bins = pd.cut(
    df["listing_age_days"],
    bins=[-1, 1, 7, 30, 90, 180, 10000],
    labels=["≤1 day", "1-7 days", "1-4 weeks", "1-3 months", "3-6 months", "6+ months"],
)
age_dist = age_bins.value_counts().sort_index().reset_index()
age_dist.columns = ["age_bucket", "count"]

fig = px.bar(
    age_dist,
    x="age_bucket",
    y="count",
    color="count",
    color_continuous_scale="OrRd",
    title="Listing age distribution",
    text="count",
)
fig.update_traces(textposition="outside")
fig.update_layout(height=400, showlegend=False)
fig.show()

print(f"\nMedian listing age: {df['listing_age_days'].median():.0f} days")
print(f"% listings older than 6 months: {(df['listing_age_days'] > 180).mean()*100:.1f}%")
print(f"% listings with price drop: {df['price_dropped'].mean()*100:.1f}%")

## 9. Correlation matrix

Linear (Pearson) correlations between numeric features and price.

In [ ]:
numeric_cols = [
    "price_eur", "price_per_m2", "surface_m2", "bedrooms", "bathrooms", "floor",
    "distance_to_valencia_km", "distance_to_coast_km",
    "feature_count", "has_elevator", "has_parking", "has_terrace",
    "has_pool", "has_garden", "has_air_conditioning",
]
corr = df[numeric_cols].corr(numeric_only=True).round(2)

fig = px.imshow(
    corr,
    text_auto=True,
    aspect="auto",
    color_continuous_scale="RdBu_r",
    color_continuous_midpoint=0,
    title="Correlation matrix (numeric features)",
)
fig.update_layout(height=700)
fig.show()

## 10. Bargain hunting — €/m² leaderboards

Cheapest 10 listings by €/m² in coastal & metro regions (sanity-checked: ≥50 m², price ≥ €50k).

In [ ]:
def bargain_table(df_subset, n=10):
    return (
        df_subset[df_subset["surface_m2"] >= 50]
        [df_subset["price_eur"] >= 50000]
        .nsmallest(n, "price_per_m2")
        [["property_subtype", "city", "district", "price_eur", "surface_m2",
          "price_per_m2", "bedrooms", "bathrooms", "feature_count", "url"]]
        .reset_index(drop=True)
    )

print("=== Cheapest €/m² in Valencia Capital ===")
display(bargain_table(df[df["region"] == "Valencia Capital"]))
print("\n=== Cheapest €/m² on the Coast ===")
display(bargain_table(df[df["region"] == "Coast"]))

## 11. Key insights

After running all cells above, capture findings here. (See `INSIGHTS.md` for the curated summary.)

The next step is **Phase 3 — price prediction model**, which will let us flag listings priced significantly below their predicted fair value.